<div align="center">

# Predicción de tasas de mortalidad y natalidad a partir de indicadores socioeconómicos

**Proyecto Integrador**

Universidad Internacional del Ecuador — UIDE

---

**Notebook 04 — Validación metodológica**

---

Equipo: Guillermo Paredes · Donato Oña · Mateo Villacreses

Junio 2026

</div>

## Objetivo

Cerrar el ciclo metodológico del proyecto con dos propósitos:

1. **Verificar la integración del pipeline** `00 → 01 → 02 → 03`: confirmar que los artefactos que produce cada notebook existen, son consistentes y alimentan correctamente al siguiente. Esta celda de verificación es el **chequeo formal** que debe pasar antes de integrar nuevos cambios a `main`.
2. **Documentar y analizar los tres problemas metodológicos** que condicionan la interpretación de los resultados:
   - **P1.** El alcance del coeficiente de Gini (solo desigualdad de ingresos).
   - **P2.** La naturaleza del *ground truth* de las tasas demográficas (estimaciones modeladas, no registro civil crudo).
   - **P3.** El tratamiento de outliers **por tipo de variable**, no mediante un IQR uniforme.

## Estructura del notebook

1. Verificación de integración del pipeline.
2. Problema metodológico 1 — Alcance del coeficiente de Gini.
3. Problema metodológico 2 — *Ground truth* de las tasas demográficas.
4. Problema metodológico 3 — Outliers por tipo de variable.
5. Conclusiones de la validación.

## 1. Verificación de integración del pipeline

Esta sección comprueba que la cadena completa de notebooks produce artefactos consistentes. Carga las salidas de los notebooks 01 (crudo) y 02 (procesado) y valida un conjunto de **invariantes**: dependencias declaradas, existencia de los Parquet, esquema de columnas esperado, ausencia de valores faltantes en los features, filtrado de agregados y cobertura plausible de países y años. Si alguna verificación falla, la celda final del bloque lanza una excepción.

> Requiere haber ejecutado previamente los notebooks 01 y 02 (que regeneran `data/`, no versionada).

In [ ]:
import os
import numpy as np
import pandas as pd

RAW_PATH = '../data/raw/worldbank_indicators.parquet'
PROCESSED_PATH = '../data/processed/dataset_modelado.parquet'

errores = []
def check(cond, msg):
    print(f"  [{'OK   ' if cond else 'FALLA'}] {msg}")
    if not cond:
        errores.append(msg)

print("=== VERIFICACIÓN DE INTEGRACIÓN DEL PIPELINE ===\n")

# 1) Dependencias declaradas (resuelto el conflicto C1)
req = open('../requirements.txt', encoding='utf-8').read()
check('wbgapi' in req, "requirements.txt declara wbgapi")
check('pyarrow' in req, "requirements.txt declara pyarrow")

# 2) Artefacto del notebook 01 (crudo)
check(os.path.exists(RAW_PATH), f"existe el crudo del 01 ({RAW_PATH})")

# 3) Artefacto del notebook 02 (procesado) - puente 02->03 (resuelto C3)
check(os.path.exists(PROCESSED_PATH), f"existe el procesado del 02 ({PROCESSED_PATH})")
assert os.path.exists(PROCESSED_PATH), (
    "Falta el dataset procesado. Ejecuta el notebook 02 (Run All) antes de este."
)
df = pd.read_parquet(PROCESSED_PATH)

# 4) Esquema e invariantes del dataset procesado
META = ['country_code', 'country_name', 'region', 'income_level', 'year']
TARGETS = ['death_rate', 'birth_rate']
features = [c for c in df.columns if c not in META + TARGETS]

check(set(TARGETS).issubset(df.columns), "targets presentes (death_rate, birth_rate)")
check('is_pandemic' in df.columns, "flag is_pandemic presente")
check('gini_imputed' in df.columns, "flag gini_imputed presente")
check(any(c.startswith('log_') for c in df.columns), "transformaciones log presentes")
check(any(c.startswith('region_') for c in df.columns), "one-hot de región presente")
check('income_level_ord' in df.columns, "income_level_ord presente")
check(df[features].isna().sum().sum() == 0, "sin NaN en features tras imputación")
check('region_' not in df.columns, "sin columna 'region_' (agregados filtrados, C2)")

# 5) Cobertura plausible (agregados filtrados -> ~200-220 países; años 2000-2022)
n_paises = df['country_code'].nunique()
check(180 <= n_paises <= 230, f"nº de países en rango plausible ({n_paises})")
check(bool(df['year'].between(2000, 2022).all()),
      f"años dentro de 2000-2022 ({df['year'].min()}-{df['year'].max()})")

print(f"\nShape procesado: {df.shape} | países: {n_paises} | features: {len(features)}")

In [ ]:
# Sanity check de modelado: el dataset entrena de extremo a extremo bajo GroupKFold
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GroupKFold, cross_val_score

X = df[features].values
groups = df['country_code'].values
modelo = Pipeline([('sc', StandardScaler()), ('reg', Ridge(alpha=1.0))])
r2 = cross_val_score(modelo, X, df['birth_rate'].values, groups=groups,
                     cv=GroupKFold(5), scoring='r2')
check(np.isfinite(r2).all(), "el modelo entrena y produce R² finito bajo GroupKFold")
print(f"R² (Ridge, birth_rate, GroupKFold 5): media={r2.mean():.3f}")

print("\n" + ("✅ INTEGRACIÓN OK - el pipeline 00->01->02->03 es consistente."
              if not errores else
              f"❌ {len(errores)} verificación(es) fallaron: {errores}"))
assert not errores, "La verificación de integración encontró problemas (ver detalle arriba)."

## 2. Problema metodológico 1 — Alcance del coeficiente de Gini

El proyecto emplea **`SI.POV.GINI`**, que mide **exclusivamente la desigualdad de ingresos (o consumo) entre hogares**. Es importante delimitar su alcance para no sobre-interpretarlo:

- **No mide** desigualdad de salud, de acceso a servicios, de género ni de riqueza (patrimonio). Un país puede tener un Gini de ingresos moderado y, aun así, fuertes inequidades sanitarias.
- Su relación con la mortalidad/natalidad es, por tanto, **parcial e indirecta**: capta una dimensión de la desigualdad, no la desigualdad en salud que más directamente incidiría en las tasas.
- **Cobertura muy baja**: proviene de encuestas de hogares puntuales, no de un registro continuo. En el crudo su cobertura ronda el 29% y en el dataset de modelado la mayoría de valores son **imputados** (bandera `gini_imputed`).

**Implicación.** La importancia estimada del Gini debe leerse con cautela: (a) por su alcance acotado a ingresos y (b) por su alta imputación. La bandera `gini_imputed` permite al modelo —y al analista— distinguir observaciones reportadas de imputadas.

In [ ]:
raw = pd.read_parquet(RAW_PATH)

print("Alcance y cobertura del coeficiente de Gini (SI.POV.GINI - solo ingresos):\n")
print(f"  Cobertura observada de gini en el crudo     : {raw['gini'].notna().mean()*100:5.1f}%")
print(f"  Fracción imputada en el dataset de modelado : {df['gini_imputed'].mean()*100:5.1f}%")
print(f"  Rango de gini (procesado)                   : {df['gini'].min():.1f}-{df['gini'].max():.1f}")
print("\nNota: el Gini NO mide desigualdad de salud, género, acceso ni riqueza patrimonial.")

## 3. Problema metodológico 2 — *Ground truth* de las tasas demográficas

Las variables objetivo (`death_rate`, `birth_rate`) **no son mediciones crudas de registro civil**: el Banco Mundial publica **estimaciones modeladas** (interpoladas/proyectadas a partir de censos, encuestas y sistemas de registro de calidad heterogénea). Muchos países carecen de registro civil completo, por lo que las cifras son el resultado de un modelo demográfico subyacente.

**Consecuencias para nuestro modelo:**

- En la práctica predecimos, en parte, **estimaciones de estimaciones**: el *ground truth* ya está suavizado y tiene su propia incertidumbre.
- Esto impone un **techo** al desempeño alcanzable y explica parte de los residuales irreducibles (especialmente en países con registro débil).
- La **cobertura casi total** de las tasas (≈100%) frente a la baja cobertura de indicadores encuestados (Gini, desempleo) es, en sí misma, evidencia de que las tasas están modeladas: un indicador genuinamente observado y disperso no tendría cobertura completa.

In [ ]:
print("Cobertura observada en el crudo - contraste 'modelado' vs 'encuestado':\n")
modeladas = ['death_rate', 'birth_rate', 'population', 'urban_pop_pct']
encuestadas = ['gini', 'unemployment', 'edu_exp_pct_gdp']
for col in modeladas + encuestadas:
    tipo = 'modelada/estimada' if col in modeladas else 'encuestada/reportada'
    print(f"  {col:18s} {raw[col].notna().mean()*100:5.1f}%   ({tipo})")
print("\nLa cobertura ~completa de las tasas refleja su carácter de estimación modelada.")

## 4. Problema metodológico 3 — Outliers por tipo de variable

Aplicar un criterio **uniforme** de outliers (p. ej. IQR ±1.5 a todas las columnas) es metodológicamente incorrecto en este dataset, porque cada variable tiene un **dominio y una forma distintos**:

- **Tasas** (`death_rate`, `birth_rate`): no negativas; un valor `< 0` es imposible (error), pero una natalidad de 50‰ en el Sahel es **legítima**, no un outlier.
- **Porcentajes** (`urban_pop_pct`, estructura etaria, gasto %PIB): acotados a `[0, 100]`; lo fuera de rango es error de dominio.
- **Índices** (`gini`): acotado a `[0, 100]`.
- **Crecimiento del PIB** (`gdp_growth`): **puede ser negativo**; las recesiones (valores muy bajos) son datos válidos, no ruido.
- **Variables económicas sesgadas** (`gdp_per_capita`, `population`): cola larga legítima → se tratan con **transformación logarítmica**, no recortando los extremos.

**Criterio adoptado:** detectar y corregir solo **violaciones de dominio** (valores imposibles) y manejar la asimetría con transformaciones; **no** descartar valores extremos pero plausibles. La celda siguiente cuantifica las violaciones de dominio y muestra cómo un IQR uniforme **sobre-marcaría** observaciones perfectamente válidas.

In [ ]:
# Dominios plausibles por tipo de variable
dominios = {
    'death_rate':         (0, 60),    # tasa por 1.000 hab., >= 0
    'birth_rate':         (0, 60),
    'urban_pop_pct':      (0, 100),   # porcentaje
    'gini':               (0, 100),   # índice 0-100
    'health_exp_pct_gdp': (0, 100),
    'edu_exp_pct_gdp':    (0, 100),
    'unemployment':       (0, 100),
    'gdp_per_capita':     (0, None),  # >= 0, sesgada -> log (no recortar)
    'population':         (0, None),
    # gdp_growth: SIN cota inferior -> las recesiones (negativos) son válidas
}

print("a) Violaciones de DOMINIO (valores imposibles, no meramente extremos):\n")
total_viol = 0
for col, (lo, hi) in dominios.items():
    s = raw[col]
    viol = (s < lo) if hi is None else ((s < lo) | (s > hi))
    total_viol += int(viol.sum())
    print(f"   {col:20s} dominio [{lo}, {hi}] -> {int(viol.sum())} violaciones")
print(f"\n   Total de valores fuera de dominio: {total_viol}")

print("\nb) Un IQR UNIFORME (k=1.5) sobre-marca valores legítimos como 'outliers':\n")
for col in ['birth_rate', 'gdp_growth', 'gdp_per_capita']:
    s = raw[col].dropna()
    q1, q3 = s.quantile([.25, .75]); iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n = int(((s < lo) | (s > hi)).sum())
    print(f"   {col:16s}: IQR marcaría {n:4d} ({n/len(s)*100:4.1f}%) como outliers, "
          f"banda [{lo:.1f}, {hi:.1f}]")
print("\n   -> Incluyen natalidades altas reales, recesiones y países ricos: NO deben descartarse.")

## 5. Conclusiones de la validación

- **Integración del pipeline:** la cadena `00 → 01 → 02 → 03` queda verificada por una celda de chequeos que se ejecuta sin fallos. Esta celda es el criterio formal a satisfacer antes de integrar cambios futuros a `main`.
- **P1 — Alcance del Gini:** se interpreta estrictamente como desigualdad de **ingresos**, con cobertura baja y fuerte imputación; su importancia se lee con la cautela que señala la bandera `gini_imputed`.
- **P2 — Ground truth:** las tasas son **estimaciones modeladas** del Banco Mundial, no registro civil crudo; esto impone un techo al desempeño y contextualiza los residuales irreducibles.
- **P3 — Outliers por tipo:** se corrigen solo **violaciones de dominio** y se maneja la asimetría con **logaritmos**; se evita el IQR uniforme, que descartaría valores extremos pero válidos (natalidades altas, recesiones, países ricos).

Estas tres consideraciones acompañan la lectura de las métricas del notebook 03 y enmarcan honestamente el alcance del proyecto.